# Install Dependencies

In [ ]:
!pip install -q -U accelerate=='0.25.0' peft=='0.7.1' bitsandbytes=='0.41.3.post2' transformers=='4.36.1' trl=='0.7.4

In [ ]:
!pip install -q bitsandbytes
!pip install -q datasets
!pip install -q peft
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 12.1 MB/s eta 0:00:00


# login

In [ ]:
import huggingface_hub
huggingface_hub.login(Hugging_Face_TOKEN)

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import random
import pandas as pd
import re
import string
import sys
import argparse
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
from sklearn.utils import shuffle
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import emoji
import pyarabic.araby as araby

# Define Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['إيجابية', 'سلبية', 'محايدة']
    mapping = {'إيجابية': 0, 'سلبية': 1, 'محايدة':2, 'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred, digits = 4)
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2, 3])
    print('\nConfusion Matrix:')
    print(conf_matrix)

# Define Predict Function

In [ ]:
def predict(X_test, model, tokenizer):
    y_pred = []
    max_length = tokenizer.model_max_length
    for i in tqdm(range(len(X_test))):
        prompt = X_test.iloc[i]["text"]
        result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
        answer = result[0]['generated_text'].split("=")[-1].lower()
        if ("سلبية" in answer
            or "سلبي" in answer
            or "سلبيّة" in answer):
            y_pred.append("سلبية")
        elif ("إيجابية" in answer
              or "إيجابي" in answer
              or "إيجابيّة" in answer):
            y_pred.append("إيجابية")
        elif ("محايدة" in answer
              or "محايد" in answer):
            y_pred.append("محايدة")
        else:
            y_pred.append("none")
    return y_pred

# Load the Data

In [ ]:
data = pd.read_excel('All Climate Change Data - All Related.xlsx')

data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True, subset='text')
data.reset_index(drop=True, inplace = True)

data.shape

(23957, 4)

In [ ]:
data.head()

,location,date,text,sentiment
0,NaN,"2008, 10, 27, 17, 41, 17, tzinfo=datetime.time...",هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative
1,Türkiye,"2023, 2, 20, 19, 2, 5, tzinfo=datetime.timezon...",#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Neutral
2,NaN,"2023, 2, 17, 9, 45, tzinfo=datetime.timezone.utc",RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Neutral
3,NaN,"2024, 8, 17, 16, 1, 17, tzinfo=datetime.timezo...",رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Positive
4,NaN,"2024, 8, 11, 10, 32, 31, tzinfo=datetime.timez...",قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


# Preprocessing

In [ ]:
data['text'] = data['text'].str.replace(r'[^\w\s]+', '')
data['text'] = data['text'].str.replace("\s+", " ", regex=True)

data.head()

,location,date,text,sentiment
0,NaN,"2008, 10, 27, 17, 41, 17, tzinfo=datetime.time...",هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative
1,Türkiye,"2023, 2, 20, 19, 2, 5, tzinfo=datetime.timezon...",#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Neutral
2,NaN,"2023, 2, 17, 9, 45, tzinfo=datetime.timezone.utc",RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Neutral
3,NaN,"2024, 8, 17, 16, 1, 17, tzinfo=datetime.timezo...",رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Positive
4,NaN,"2024, 8, 11, 10, 32, 31, tzinfo=datetime.timez...",قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
data = data[['text', 'sentiment']]

In [ ]:
data['sentiment'].value_counts()

,count
sentiment,
Neutral,10307
Negative,8140
Positive,5510


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
data['text'] = data['text'].apply(normalize_arabic)

data.head()

,text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative
1,#عاجل | ادارة الكوارث والطوارئ التركية: ازالة ...,Neutral
2,RT @USUN: عُقد في مالطا هذا الاسبوع اول اجتماع...,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Fix elongated words
    pattern = re.compile(r'(.)\1{2,}')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=r'\1'))
    # Normalize letters
    pattern = re.compile("[إأآا]")
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl="ا"))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
data['text'] = data_cleaning(data['text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

data['text'] = data['text'].apply(remove_ids)
data.head()

,text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Neutral
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
data.isnull().sum()

,0
text,0
sentiment,0


In [ ]:
# Mapping dictionary
sentiment_map = {
    'Positive': 'إيجابية',
    'Negative': 'سلبية',
    'Neutral': 'محايدة'
}

# Apply the mapping
data['Final Label'] = data['sentiment'].map(sentiment_map)
data.head()

,text,sentiment,Final Label
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative,سلبية
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Neutral,محايدة
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Neutral,محايدة
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Positive,إيجابية
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral,محايدة


In [ ]:
data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True)
data.shape

# Split Data

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate.csv')

# Clean and convert to integer index arrays
test = data.loc[indecies['test'].dropna().astype(int).values]
train = data.loc[indecies['train'].dropna().astype(int).values]
val = data.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train = train.reset_index(drop=True)

def generate_prompt(data_point):
    return f"""
    [INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر المتعلقة بتغير المناخ.
    قم بتصنيف مشاعر الجملة العربية التالية  بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية تجاه تغير المناخ. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.  [/INST]
    [{data_point["text"]}] = [{data_point["Final Label"]}]
            """.strip()

def generate_test_prompt(data_point):
    return f"""
    أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر المتعلقة بتغير المناخ.
    قم بتصنيف مشاعر الجملة العربية التالية بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية تجاه تغير المناخ. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.
    [{data_point["text"]}] =
            """.strip()

train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["text"])
val = pd.DataFrame(val.apply(generate_prompt, axis=1),
                      columns=["text"])

In [ ]:
y_true = test["Final Label"]
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["text"])

train_data = Dataset.from_pandas(train)
eval_data = Dataset.from_pandas(train)

# Load Model

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Predict Without Fine-tuning

In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

# Fine-tune Model

In [ ]:
from trl import SFTConfig

peft_config = LoraConfig(
    lora_alpha=128,
    lora_dropout=0.05,
    r=64,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = SFTConfig(
    output_dir="logs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1, # 4
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0,
    fp16=False,
    bf16=False,
    max_grad_norm=1,
    max_steps=-1,
    warmup_ratio=0.01,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    eval_strategy="epoch",
    dataset_text_field="text",
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained("/content/drive/MyDrive/Allam")

Epoch,Training Loss,Validation Loss
1,1.565700,1.299328


Epoch,Training Loss,Validation Loss
1,1.565700,1.299328
2,1.190500,0.785494
3,0.671300,0.450172


# Predict

In [ ]:
from peft import PeftModel, PeftConfig
# Load the Lora model
model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/Allam')

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=10,
                temperature=0.2
               )

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

100%|██████████| 3594/3594 [1:01:59<00:00,  1.03s/it]

Accuracy: 0.525

f1_score:  0.551000504984853

Classification Report:
              precision    recall  f1-score   support

           0     0.4851    0.7473    0.5883       827
           1     0.6949    0.5315    0.6023      1221
           2     0.6298    0.4017    0.4905      1546
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.5253      3594
   macro avg     0.4524    0.4201    0.4203      3594
weighted avg     0.6186    0.5253    0.5510      3594


Confusion Matrix:
[[618  14 144  51]
 [211 649 221 140]
 [445 271 621 209]
 [  0   0   0   0]]


# Pred on Asad

In [ ]:
asad2 = pd.read_csv('train_all.csv')

asad2['Text'] = asad2['Text'].str.replace(r'[^\w\s]+', '')
asad2['Text'] = asad2['Text'].str.replace("\s+", " ", regex=True)

asad2['Text'] = asad2['Text'].apply(normalize_arabic)

asad2['Text'] = data_cleaning(asad2['Text'])
asad2['Text'] = asad2['Text'].apply(remove_ids)

asad2 = asad2.rename(columns={'Text': 'text'})
asad2.head()

,Tweet_id,sentiment,text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...
1,1221884400377499651,neutral,@ @ ليس حبا في ايران بقدر ماهو نكايه بترامب وحزبه
2,1221881406168731649,neutral,@ ابي اعرف الحاكم العربي المسلم اشلون ينام ماي...
3,1221882047691657217,neutral,@ @ في الخطاب تبع سليم سعاده حطت عالتويتر شو ق...
4,1221880371773673474,neutral,@ مفيش الكلام ده في الزمن


In [ ]:
asad2['sentiment'] = asad2['sentiment'].map({
    'negative': 'سلبية',
    'neutral': 'محايدة',
    'positive': 'إيجابية'
})

In [ ]:
train, asad20 = train_test_split(asad2,
                               test_size=20000,
                               stratify=asad2['sentiment'],
                               random_state=42)

asad20_test = pd.DataFrame(asad20.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pred_asad20 = predict(asad20_test, pipe, tokenizer)
evaluate(asad20['sentiment'], pred_asad20)

100%|██████████| 20000/20000 [5:43:00<00:00,  1.03s/it]

Accuracy: 0.513

f1_score:  0.5563838031366657

Classification Report:
              precision    recall  f1-score   support

           0     0.3071    0.3183    0.3126      3208
           1     0.4390    0.3757    0.4049      3207
           2     0.7218    0.5907    0.6497     13585
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.5125     20000
   macro avg     0.3670    0.3212    0.3418     20000
weighted avg     0.6099    0.5125    0.5564     20000


Confusion Matrix:
[[1021  190 1728  269]
 [ 258 1205 1365  379]
 [2046 1350 8025 2164]
 [   0    0    0    0]]


# ASTD

In [ ]:
import pandas as pd


df = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])
df.head()

,text,Label
0,بعد استقالة رئيس #المحكمة_الدستورية ننتظر استق...,OBJ
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,POS
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,NEG
3,#الحرية_والعدالة | شاهد الآن: #ليلة_الاتحادية ...,OBJ
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,NEUTRAL


In [ ]:
df = df[df['Label']!='OBJ']
df['Label'] = df['Label'].map({
    'NEG': 'سلبية',
    'NEUTRAL': 'محايدة',
    'POS': 'إيجابية'
})

df.head()

,text,Label
1,أهنئ الدكتور أحمد جمال الدين، القيادي بحزب مصر...,إيجابية
2,البرادعي يستقوى بامريكا مرةاخرى و يرسل عصام ال...,سلبية
4,الوالدة لو اقولها بخاطري حشيشة تضحك بس من اقول...,محايدة
5,#انتخبوا_العرص #انتخبوا_البرص #مرسى_رئيسى #اين...,محايدة
6,امير عيد هو اللي فعلا يتقال عليه ستريكر صريح #...,إيجابية


In [ ]:
df['text'] = df['text'].str.replace(r'[^\w\s]+', '')
df['text'] = df['text'].str.replace("\s+", " ", regex=True)

df['text'] = df['text'].apply(normalize_arabic)

df['text'] = data_cleaning(df['text'])
df['text'] = df['text'].apply(remove_ids)

df.dropna(inplace = True)
df = df.drop_duplicates(subset = 'text')

astd_test = pd.DataFrame(df.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
max_length = tokenizer.model_max_length
for i in tqdm(range(len(astd_test[:5]))):
    prompt = astd_test.iloc[i]["text"]
    result = pipe(prompt[:tokenizer.model_max_length], truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("=")[-1].lower()
    print(answer)

 20%|██        | 1/5 [00:01<00:04,  1.18s/it]

    [محايدة] 


 40%|████      | 2/5 [00:02<00:03,  1.04s/it]

    سلبية. 


 60%|██████    | 3/5 [00:03<00:02,  1.09s/it]

    [محايدة] 


 80%|████████  | 4/5 [00:04<00:01,  1.12s/it]

    محايدة  




100%|██████████| 5/5 [00:05<00:00,  1.07s/it]

    محايدة 


In [ ]:
pred_astd = predict(astd_test, pipe, tokenizer)
evaluate(df['Label'], pred_astd)

100%|██████████| 3223/3223 [56:27<00:00,  1.05s/it]

Accuracy: 0.467

f1_score:  0.49463538342964125

Classification Report:
              precision    recall  f1-score   support

           0     0.5042    0.3822    0.4348       777
           1     0.8004    0.4449    0.5719      1641
           2     0.2955    0.5950    0.3949       805
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.4673      3223
   macro avg     0.4000    0.3555    0.3504      3223
weighted avg     0.6029    0.4673    0.4946      3223


Confusion Matrix:
[[297  27 426  27]
 [154 730 716  41]
 [138 155 479  33]
 [  0   0   0   0]]


# SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'سلبية',
    'neutral': 'محايدة',
    'positive': 'إيجابية'
})
semeval.head()

,text,sentiment
0,إجبار أبل على التعاون على فك شفرة اجهزتها http...,إيجابية
1,RT @20fourMedia: #غوغل تتحدى أبل وأمازون بأجهز...,إيجابية
2,جوجل تنافس أبل وسامسونج بهاتف جديد https://t.c...,إيجابية
3,رئيس شركة أبل: الواقع المعزز سيصبح أهم من الطع...,إيجابية
4,ساعة أبل في الأسواق مرة أخرى https://t.co/dY2x...,محايدة


In [ ]:
semeval['text'] = semeval['text'].str.replace(r'[^\w\s]+', '')
semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)

semeval['text'] = semeval['text'].apply(normalize_arabic)

semeval['text'] = data_cleaning(semeval['text'])
semeval['text'] = semeval['text'].apply(remove_ids)

semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')

semeval_test = pd.DataFrame(semeval.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pred_semeval = predict(semeval_test, pipe, tokenizer)
evaluate(semeval['sentiment'], pred_semeval)

100%|██████████| 3287/3287 [58:29<00:00,  1.07s/it]

Accuracy: 0.385

f1_score:  0.41018612847096547

Classification Report:
              precision    recall  f1-score   support

           0     0.3054    0.1241    0.1765       733
           1     0.7113    0.2755    0.3971      1100
           2     0.4885    0.5983    0.5379      1454
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.3845      3287
   macro avg     0.3763    0.2495    0.2779      3287
weighted avg     0.5222    0.3845    0.4102      3287


Confusion Matrix:
[[ 91  24 450 168]
 [101 303 461 235]
 [106  99 870 379]
 [  0   0   0   0]]
